# Sea-salt bromine recycling (CESM2-SLH) with MICM

This notebook calls `quacs.vsls.cesm2_slh`. It plots the rate constants and runs the MUSICA box model.
See `quacs/vsls/cesm2_slh/README.md` for the inputs, the outputs, and the equations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from quacs.vsls.cesm2_slh import SEA_SALT_REACTIONS, run_box_model, seasalt_rate_constants

## Inputs

These are illustrative marine boundary layer values, not observations.

In [ ]:
met = dict(
    temperature_k=288.0,
    pressure_pa=1.0e5,
    latitude_deg=40.0,
    calday=180.0,
    sad_seasalt_m2_m3=5.0e-5,  # 50 um2/cm3
    ocean_ice_fraction=1.0,
)
initial_pptv = {"HOBr": 5.0, "BrONO2": 2.0, "BrNO2": 0.5}

## Rate constants as a function of temperature and latitude

In [ ]:
temperature = np.linspace(240.0, 310.0, 50)
latitude = np.linspace(-90.0, 90.0, 181)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
k_t = seasalt_rate_constants(**{**met, "temperature_k": temperature})
k_lat = seasalt_rate_constants(**{**met, "latitude_deg": latitude, "calday": 1.0})
for rxn in SEA_SALT_REACTIONS:
    ax1.plot(temperature, k_t[rxn.name], label=f"{rxn.name} ({rxn.reactant})")
    ax2.plot(latitude, k_lat[rxn.name], label=f"{rxn.name} ({rxn.reactant})")
ax1.set_xlabel("Temperature (K)")
ax1.set_ylabel("k (1/s)")
ax2.set_xlabel("Latitude (degrees north), calday = 1")
ax2.set_ylabel("k (1/s)")
ax1.legend()
plt.tight_layout()

## MUSICA box model, 24 h

In [ ]:
out = run_box_model(met, initial_pptv, time_step_s=600.0, n_steps=144)

fig, ax = plt.subplots(figsize=(6, 4))
hours = out["times"] / 3600.0
for name, conc in out["concentrations"].items():
    ax.plot(hours, conc / out["n_air"] * 1e12, label=name)
ax.set_xlabel("Time (h)")
ax.set_ylabel("Mixing ratio (pptv)")
ax.legend()
plt.tight_layout()